In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
from datetime import timedelta

# Load data
data = pd.read_csv("GOOG_C_cleaned.csv", parse_dates=['Date'])
data.set_index('Date', inplace=True)
prices = data['Close']

# Compute log returns
log_returns = np.log(prices / prices.shift(1)).dropna()

# Fit GARCH(1,1) and forecast 5-day σₜ
model = arch_model(log_returns, vol='Garch', p=1, q=1)
garch_fit = model.fit(disp='off')
vol_forecast = np.sqrt(garch_fit.forecast(horizon=5).variance.values[-1])

# Monte Carlo simulate returns
n_sim = 10000
horizon = 5
sim_returns = np.zeros((n_sim, horizon))
for t in range(horizon):
    sim_returns[:, t] = vol_forecast[t] * np.random.normal(size=n_sim)

# Reconstruct price paths
S0 = prices.iloc[-1]
sim_prices = S0 * np.exp(np.cumsum(sim_returns, axis=1))

# Build and save forecast
forecast_dates = [prices.index[-1] + timedelta(days=i+1) for i in range(horizon)]
forecast_df = pd.DataFrame({
    'MedianPrice': np.median(sim_prices, axis=0),
    '5thPercentile': np.percentile(sim_prices, 5, axis=0),
    '95thPercentile': np.percentile(sim_prices, 95, axis=0),
}, index=forecast_dates)
forecast_df.to_csv("GOOG_MC_GARCH_returns_forecast.csv")


/Users/alexanderdagher/.pyenv/versions/3.10.12/lib/python3.10/site-packages/arch/univariate/base.py:309: DataScaleWarning: y is poorly scaled, which may affect convergence of the optimizer when
estimating the model parameters. The scale of y is 0.0003992. Parameter
estimation work better when this value is between 1 and 1000. The recommended
rescaling is 100 * y.

This warning can be disabled by either rescaling y before initializing the
model or by setting rescale=False.

  warnings.warn(
